In [1]:
import os
import torch
from torch import nn
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
os.chdir('../')

In [3]:
from src.LogisticRegression import LogisticRegression

### Pipeline de Treinamento

1. Modelo
2. Loss e Otimizador
3. Loop de Treinamento
    - Forward Pass
    - Calcula Loss
    - Backward Pass
    - Atualiza Pesos

In [4]:
# 0. Preparação de dados
dataset = datasets.load_breast_cancer()

In [5]:
dataset.feature_names

array(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error',
       'fractal dimension error', 'worst radius', 'worst texture',
       'worst perimeter', 'worst area', 'worst smoothness',
       'worst compactness', 'worst concavity', 'worst concave points',
       'worst symmetry', 'worst fractal dimension'], dtype='<U23')

In [6]:
dataset.target_names

array(['malignant', 'benign'], dtype='<U9')

In [7]:
X_data = dataset.data    # Features (matriz com as colunas atributos e linhas)

In [8]:
y_data = dataset.target  # Label (vetor com a coluna alvo e classes)

In [9]:
print(y_data)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 0 0 1 0 0 1 1 1 1 0 1 0 0 1 1 1 1 0 1 0 0
 1 0 1 0 0 1 1 1 0 0 1 0 0 0 1 1 1 0 1 1 0 0 1 1 1 0 0 1 1 1 1 0 1 1 0 1 1
 1 1 1 1 1 1 0 0 0 1 0 0 1 1 1 0 0 1 0 1 0 0 1 0 0 1 1 0 1 1 0 1 1 1 1 0 1
 1 1 1 1 1 1 1 1 0 1 1 1 1 0 0 1 0 1 1 0 0 1 1 0 0 1 1 1 1 0 1 1 0 0 0 1 0
 1 0 1 1 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 1 0 1 1 0 1 0 0 0 0 1 1 0 0 1 1
 1 0 1 1 1 1 1 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 0 1 1 1 1 1 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 1 1 1 1 0 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 0 1 1 1 1 0 0 0 1 1
 1 1 0 1 0 1 0 1 1 1 0 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 0
 0 1 0 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1
 1 0 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 1 0 1 1 1 1 1 0 1 1
 0 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1
 1 1 1 1 1 1 0 1 0 1 1 0 

In [10]:
class0, class1 = [], []
for x in y_data:
    if x == 0:
        class0.append(x)
    elif x == 1:
        class1.append(x)

In [11]:
print(len(class0), len(class1))

212 357


In [12]:
print(type(X_data), type(y_data))

<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [13]:
print(X_data.shape, y_data.shape)

(569, 30) (569,)


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.3, random_state=42
)

In [15]:
X_train_torch = torch.from_numpy(X_train.astype(np.float32))
X_test_torch = torch.from_numpy(X_test.astype(np.float32))
y_train_torch = torch.from_numpy(y_train.astype(np.float32))
y_test_torch = torch.from_numpy(y_test.astype(np.float32))

In [16]:
# 1. Modelo
model = LogisticRegression(n_input=len(dataset.feature_names))

In [17]:
# 2. Loss e Otimizador
learning_rate = 0.001
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [18]:
# 3. Loop de Treino
num_epochs = 100
for epoch in range(num_epochs):
    # 3.1. Forward Pass
    y_pred = model(X_train_torch)

    # 3.2. Cálculo da Loss
    loss = criterion(y_pred, y_train_torch.view(-1, 1))  # Compara predição com dado real

    # 3.3. Backward Pass
    loss.backward()

    # 3.4. Updates
    optimizer.step()
    optimizer.zero_grad()  # Zero gradients

### Avaliação

In [19]:
with torch.no_grad():
    y_pred = model(X_test_torch)  # Predição de teste

    # Métricas de avaliação: comparações entre previsão e real
    y_pred_round = y_pred.round()
    print(accuracy_score(y_pred_round, y_test_torch))

0.3684210526315789
